# Day 28: Decision Tree Classifier

**Dataset:** `Iris_Dataset.csv`

This notebook trains a decision tree to classify Iris species and explores how changing the tree depth affects its performance.

## 1. Import Libraries and Load the Dataset

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, plot_tree

dataset_candidates = [
    Path("../datasets/Iris_Dataset.csv"),
    Path("datasets/Iris_Dataset.csv"),
]
dataset_path = next((path for path in dataset_candidates if path.is_file()), None)
if dataset_path is None:
    raise FileNotFoundError("Could not find Iris_Dataset.csv in the datasets folder.")

iris = pd.read_csv(dataset_path)
iris.head()

## 2. Prepare the Data and Train a Decision Tree

I used the four sepal and petal measurements as features and `target` as the species label. I kept the classes balanced in both sets by using a stratified train-test split. I did not scale the measurements because decision trees split on feature thresholds.

In [ ]:
feature_columns = [
    "sepal length (cm)",
    "sepal width (cm)",
    "petal length (cm)",
    "petal width (cm)",
]

X = iris[feature_columns]
y = iris["target"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

model = DecisionTreeClassifier(max_depth=3, random_state=42)
model.fit(X_train, y_train)

print("Model trained.")
print(f"Training rows: {len(X_train)}")
print(f"Testing rows: {len(X_test)}")

**Comparing different tree depths**

I tested several values for `max_depth` and compared training accuracy with testing accuracy. A large gap between the two can be a sign that the tree is overfitting the training data.

In [ ]:
depth_options = [1, 2, 3, 4, 5, None]
depth_results = []

for max_depth in depth_options:
    tree = DecisionTreeClassifier(max_depth=max_depth, random_state=42)
    tree.fit(X_train, y_train)

    train_accuracy = accuracy_score(y_train, tree.predict(X_train))
    test_accuracy = accuracy_score(y_test, tree.predict(X_test))
    depth_results.append(
        {
            "max_depth": "Unlimited" if max_depth is None else str(max_depth),
            "train_accuracy": train_accuracy,
            "test_accuracy": test_accuracy,
            "train_test_gap": train_accuracy - test_accuracy,
            "leaf_nodes": tree.get_n_leaves(),
        }
    )

depth_comparison = pd.DataFrame(depth_results)
display(depth_comparison.round(3))

In [ ]:
plot_data = depth_comparison.melt(
    id_vars="max_depth",
    value_vars=["train_accuracy", "test_accuracy"],
    var_name="dataset",
    value_name="accuracy",
)

plt.figure(figsize=(8, 5))
sns.lineplot(
    data=plot_data,
    x="max_depth",
    y="accuracy",
    hue="dataset",
    marker="o",
    sort=False,
)
plt.title("Training and Testing Accuracy by Tree Depth")
plt.xlabel("Maximum depth")
plt.ylabel("Accuracy")
plt.ylim(0, 1.05)
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

**Observation:** Compare the two accuracy columns and the train-test gap in the table. If training accuracy keeps increasing while testing accuracy levels off or drops, the deeper tree is fitting the training examples better without improving its predictions on the held-out data.

## 3. Confusion Matrix and Classification Report

I used the depth-3 model to review the predictions by species. The confusion matrix shows the number of correct and incorrect predictions, while the classification report summarizes precision, recall, and F1-score for each class.

In [ ]:
y_pred = model.predict(X_test)

print(f"Test accuracy: {accuracy_score(y_test, y_pred):.3f}")
print("\nClassification report:")
print(classification_report(y_test, y_pred, zero_division=0))

In [ ]:
class_labels = model.classes_
matrix = confusion_matrix(y_test, y_pred, labels=class_labels)

plt.figure(figsize=(7, 5))
sns.heatmap(
    matrix,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=class_labels,
    yticklabels=class_labels,
)
plt.title("Confusion Matrix for the Iris Decision Tree")
plt.xlabel("Predicted species")
plt.ylabel("Actual species")
plt.tight_layout()
plt.show()

## 4. Decision Tree Visualization and Feature Importance

The tree visualization shows the feature and threshold used at each split. I also compared the feature importances to see which measurements the fitted tree relied on most.

In [ ]:
plt.figure(figsize=(15, 8))
plot_tree(
    model,
    feature_names=feature_columns,
    class_names=model.classes_,
    filled=True,
    rounded=True,
    impurity=False,
    fontsize=9,
)
plt.title("Iris Decision Tree (Maximum Depth = 3)")
plt.tight_layout()
plt.show()

importance = pd.Series(
    model.feature_importances_, index=feature_columns
).sort_values(ascending=False)
display(importance.rename("importance").to_frame())

plt.figure(figsize=(8, 4))
sns.barplot(x=importance.values, y=importance.index, color="steelblue")
plt.title("Feature Importance")
plt.xlabel("Importance")
plt.ylabel("Measurement")
plt.tight_layout()
plt.show()

## Outcome

The depth comparison shows how tree complexity changes the model's training and testing performance. The confusion matrix and classification report show which species the depth-3 tree predicted correctly, and the tree visualization makes its split rules easier to follow. The feature-importance results show which measurements contributed most to this particular model. The values and plots above are generated from the Iris data when the notebook is run.